In [1]:
import pandas as pd
import torch as th

plastchem_mol_fp = "/home/luke/fragnnet/data/proc/plastchem/mol_df.pkl"
nist20_mol_fp = "/home/luke/fragnnet/data/proc/nist20/mol_df.pkl"
nist20_spec_fp = "/home/luke/fragnnet/data/proc/nist20/spec_df.pkl"

plastchem_mol_df = pd.read_pickle(plastchem_mol_fp)
nist20_mol_df = pd.read_pickle(nist20_mol_fp)
nist20_spec_df = pd.read_pickle(nist20_spec_fp)

pred_fp = "/home/luke/fragnnet/data/inference/fragnnet_d3_195/predictions.parquet"

pred_df = pd.read_parquet(pred_fp)

In [2]:
plastchem_mol_df

,smiles,mol,mol_id,inchikey_s,scaffold,formula,inchi,mw,exact_mw,num_atoms,num_bonds,charge,single_mol,num_radicals
0,BrBr,<rdkit.Chem.rdchem.Mol object at 0x742d31a52610>,0,GDTBXPJZTBHREO,,Br2,InChI=1S/Br2/c1-2,159.808,157.836674,2,1,0,True,0
1,BrC(Br)=C(Br)c1ccccc1,<rdkit.Chem.rdchem.Mol object at 0x742d31a526b0>,1,JVPKLOPETWVKQD,c1ccccc1,C8H5Br3,InChI=1S/C8H5Br3/c9-7(8(10)11)6-4-2-1-3-5-6/h1-5H,340.840,337.794136,11,11,0,True,0
2,BrC(Br)=Cc1ccccc1,<rdkit.Chem.rdchem.Mol object at 0x742d31a52750>,2,CYLVUSZHVURAOY,c1ccccc1,C8H6Br2,InChI=1S/C8H6Br2/c9-8(10)6-7-4-2-1-3-5-7/h1-6H,261.944,259.883624,10,10,0,True,0
3,BrC(Br)Br,<rdkit.Chem.rdchem.Mol object at 0x742d2f307d80>,3,DIKBFYAXUHHXCS,,CHBr3,InChI=1S/CHBr3/c2-1(3)4/h1H,252.731,249.762836,4,3,0,True,0
4,BrC1(Br)CCCCCCCCCC(Br)(Br)C1(Br)Br,<rdkit.Chem.rdchem.Mol object at 0x742d2f305f80>,4,SHRRVNVEOIKVSG,C1CCCCCCCCCCC1,C12H18Br6,InChI=1S/C12H18Br6/c13-10(14)8-6-4-2-1-3-5-7-9...,641.700,635.650873,18,18,0,True,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9803,c1csc(C2=S=C(c3cccs3)c3nsnc32)c1,<rdkit.Chem.rdchem.Mol object at 0x742d2df34fe0>,9803,KXASLJJPAUTMIH,c1csc(C2=S=C(c3cccs3)c3nsnc32)c1,C12H6N2S4,InChI=1S/C12H6N2S4/c1-3-7(15-5-1)11-9-10(14-18...,306.462,305.941382,18,21,0,True,0
9804,c1nc(N2CCNCC2)nc(N2CCOCC2)n1,<rdkit.Chem.rdchem.Mol object at 0x742d2df35030>,9804,AVOSVDRFHAWQCC,c1nc(N2CCNCC2)nc(N2CCOCC2)n1,C11H18N6O,InChI=1S/C11H18N6O/c1-3-16(4-2-12-1)10-13-9-14...,250.306,250.154209,18,20,0,True,0
9805,c1nc[nH]n1,<rdkit.Chem.rdchem.Mol object at 0x742d2df35080>,9805,NSPMIYGKQJPBQR,c1nc[nH]n1,C2H3N3,"InChI=1S/C2H3N3/c1-3-2-5-4-1/h1-2H,(H,3,4,5)",69.067,69.032697,5,5,0,True,0
9806,c1scc2c1Cc1cscc1-2,<rdkit.Chem.rdchem.Mol object at 0x742d2df350d0>,9806,HHSONWXYOAPAQZ,c1scc2c1Cc1cscc1-2,C9H6S2,InChI=1S/C9H6S2/c1-6-2-10-4-8(6)9-5-11-3-7(1)9...,178.281,177.991092,11,13,0,True,0


In [6]:
plastchem = plastchem_mol_df[['mol_id', 'smiles', 'inchikey_s', 'mol', 'scaffold', 'formula']]
plastchem_pred_join = plastchem.merge(pred_df, on='mol_id', how='left', validate='one_to_many')
plastchem_pred_join.rename(columns={'mol_id':'plastchem_mol_id'}, inplace=True)
plastchem_pred_join

,plastchem_mol_id,smiles,inchikey_s,mol,scaffold,formula,group_id,spec_id,prec_type,inst_type,frag_mode,spec_type,ion_mode,dset,dset_spec_id,ace,prec_mz,input_peaks,pred_mzs,pred_ints
0,0,BrBr,GDTBXPJZTBHREO,<rdkit.Chem.rdchem.Mol object at 0x742d31a52610>,,Br2,0.0,0.0,[M+H]+,FT,HCD,MS2,P,inference,inference_0,20.0,158.843951,"[[1.0, 1.0]]",[160.84190368652344],[1.0]
1,1,BrC(Br)=C(Br)c1ccccc1,JVPKLOPETWVKQD,<rdkit.Chem.rdchem.Mol object at 0x742d31a526b0>,c1ccccc1,C8H5Br3,1.0,3.0,[M+H]+,FT,HCD,MS2,P,inference,inference_3,20.0,338.801413,"[[1.0, 1.0]]","[184.84190368652344, 104.93344116210938, 185.8...","[6.9508073465840425e-06, 0.0002458884264342487..."
2,2,BrC(Br)=Cc1ccccc1,CYLVUSZHVURAOY,<rdkit.Chem.rdchem.Mol object at 0x742d31a52750>,c1ccccc1,C8H6Br2,2.0,6.0,[M+H]+,FT,HCD,MS2,P,inference,inference_6,20.0,260.890901,"[[1.0, 1.0]]","[104.93344116210938, 27.022926330566406, 105.9...","[0.0003664105897769332, 2.040552089965786e-06,..."
3,3,BrC(Br)Br,DIKBFYAXUHHXCS,<rdkit.Chem.rdchem.Mol object at 0x742d2f307d80>,,CHBr3,3.0,9.0,[M+H]+,FT,HCD,MS2,P,inference,inference_9,20.0,250.770113,"[[1.0, 1.0]]","[252.76806640625, 80.93344116210938]","[0.9998470544815063, 0.00015293045726139098]"
4,4,BrC1(Br)CCCCCCCCCC(Br)(Br)C1(Br)Br,SHRRVNVEOIKVSG,<rdkit.Chem.rdchem.Mol object at 0x742d2f305f80>,C1CCCCCCCCCCC1,C12H18Br6,4.0,12.0,[M+H]+,FT,HCD,MS2,P,inference,inference_12,20.0,636.658150,"[[1.0, 1.0]]","[529.673095703125, 610.5894165039062, 530.6809...","[3.0911130579625024e-06, 2.168172386518563e-06..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9803,9803,c1csc(C2=S=C(c3cccs3)c3nsnc32)c1,KXASLJJPAUTMIH,<rdkit.Chem.rdchem.Mol object at 0x742d2df34fe0>,c1csc(C2=S=C(c3cccs3)c3nsnc32)c1,C12H6N2S4,9803.0,29409.0,[M+H]+,FT,HCD,MS2,P,inference,inference_29409,20.0,306.948659,"[[1.0, 1.0]]","[245.93746948242188, 246.94528198242188, 278.9...","[4.80210064779385e-06, 0.00022747012553736567,..."
9804,9804,c1nc(N2CCNCC2)nc(N2CCOCC2)n1,AVOSVDRFHAWQCC,<rdkit.Chem.rdchem.Mol object at 0x742d2df35030>,c1nc(N2CCNCC2)nc(N2CCOCC2)n1,C11H18N6O,9804.0,29412.0,[M+H]+,FT,HCD,MS2,P,inference,inference_29412,20.0,251.161486,"[[1.0, 1.0]]","[218.1036376953125, 219.11146545410156, 217.11...","[3.1993084121495485e-05, 2.2873149646329693e-0..."
9805,9805,c1nc[nH]n1,NSPMIYGKQJPBQR,<rdkit.Chem.rdchem.Mol object at 0x742d2df35080>,c1nc[nH]n1,C2H3N3,9805.0,29415.0,[M+H]+,FT,HCD,MS2,P,inference,inference_29415,20.0,70.039974,"[[1.0, 1.0]]","[68.02432250976562, 55.02907180786133, 69.0321...","[4.765759877045639e-05, 0.0006390911294147372,..."
9806,9806,c1scc2c1Cc1cscc1-2,HHSONWXYOAPAQZ,<rdkit.Chem.rdchem.Mol object at 0x742d2df350d0>,c1scc2c1Cc1cscc1-2,C9H6S2,9806.0,29418.0,[M+H]+,FT,HCD,MS2,P,inference,inference_29418,20.0,178.998369,"[[1.0, 1.0]]","[57.9871711730957, 27.022926330566406, 58.9949...","[0.0050027635879814625, 2.4412827315245522e-06..."


In [17]:
nist20_mol_lookup = nist20_mol_df[
    ["mol_id", "smiles", "inchikey_s"]
].rename(columns={"mol_id": "nist20_mol_id"})

plastchem_pred_nist_join = plastchem_pred_join.merge(
    nist20_mol_lookup,
    on=["smiles", "inchikey_s"],
    how="inner",
    validate="many_to_one",
)

plastchem_pred_nist_join = plastchem_pred_nist_join.merge(
    nist20_spec_df,
    left_on="nist20_mol_id",
    right_on="mol_id",
    how="inner",
    suffixes=("", "_pred_df"),
)

plastchem_pred_nist_join

,plastchem_mol_id,smiles,inchikey_s,mol,scaffold,formula,group_id,spec_id,prec_type,inst_type,...,dset_pred_df,dset_spec_id_pred_df,col_gas,res,ace_pred_df,nce,prec_mz_pred_df,peaks,ri,group_id_pred_df
0,299,C#CC(C)(C)NC(=O)c1cc(Cl)cc(Cl)c1,PHNUZKMIPFFYSO,<rdkit.Chem.rdchem.Mol object at 0x742d2f364b80>,c1ccccc1,C12H11Cl2NO,299.0,897.0,[M+H]+,FT,...,nist20_hr,1534592,N2,4,6.0,10.0,256.029,"[(67.0542, 2.2), (172.9554, 10.69), (189.9823,...",NaN,48762
1,299,C#CC(C)(C)NC(=O)c1cc(Cl)cc(Cl)c1,PHNUZKMIPFFYSO,<rdkit.Chem.rdchem.Mol object at 0x742d2f364b80>,c1ccccc1,C12H11Cl2NO,299.0,897.0,[M+H]+,FT,...,nist20_hr,1534593,N2,4,7.0,15.0,256.029,"[(67.0541, 2.8), (85.0648, 1.0), (172.9556, 15...",NaN,48762
2,299,C#CC(C)(C)NC(=O)c1cc(Cl)cc(Cl)c1,PHNUZKMIPFFYSO,<rdkit.Chem.rdchem.Mol object at 0x742d2f364b80>,c1ccccc1,C12H11Cl2NO,299.0,897.0,[M+H]+,FT,...,nist20_hr,1534594,N2,4,9.0,20.0,256.029,"[(65.0385, 1.0), (67.054, 7.09), (85.0648, 2.0...",NaN,48762
3,299,C#CC(C)(C)NC(=O)c1cc(Cl)cc(Cl)c1,PHNUZKMIPFFYSO,<rdkit.Chem.rdchem.Mol object at 0x742d2f364b80>,c1ccccc1,C12H11Cl2NO,299.0,897.0,[M+H]+,FT,...,nist20_hr,1534595,N2,4,10.0,20.0,256.029,"[(65.0385, 2.2), (67.054, 12.39), (85.0646, 2....",NaN,48762
4,299,C#CC(C)(C)NC(=O)c1cc(Cl)cc(Cl)c1,PHNUZKMIPFFYSO,<rdkit.Chem.rdchem.Mol object at 0x742d2f364b80>,c1ccccc1,C12H11Cl2NO,299.0,897.0,[M+H]+,FT,...,nist20_hr,1534596,N2,4,11.0,20.0,256.029,"[(65.0385, 2.9), (67.0541, 19.58), (85.0647, 4...",NaN,48762
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50988,9799,c1cnc2c(c1)ccc1cccnc12,DGEZNRSVGBDHLK,<rdkit.Chem.rdchem.Mol object at 0x742d2df34ea0>,c1cnc2c(c1)ccc1cccnc12,C12H8N2,9799.0,29397.0,[M+H]+,FT,...,mona23,KO003776,NaN,3,NaN,NaN,181.0,"[(39.2, 0.049516), (45.2, 0.104533), (77.0, 0....",NaN,194629
50989,9799,c1cnc2c(c1)ccc1cccnc12,DGEZNRSVGBDHLK,<rdkit.Chem.rdchem.Mol object at 0x742d2df34ea0>,c1cnc2c(c1)ccc1cccnc12,C12H8N2,9799.0,29397.0,[M+H]+,FT,...,mona23,KO003775,NaN,3,NaN,NaN,181.0,"[(42.8, 0.005007), (44.9, 0.057578), (46.1, 0....",NaN,194629
50990,9799,c1cnc2c(c1)ccc1cccnc12,DGEZNRSVGBDHLK,<rdkit.Chem.rdchem.Mol object at 0x742d2df34ea0>,c1cnc2c(c1)ccc1cccnc12,C12H8N2,9799.0,29397.0,[M+H]+,FT,...,mona23,KO003774,NaN,3,NaN,NaN,181.0,"[(38.9, 0.019122), (54.9, 0.006953), (58.8, 0....",NaN,194629
50991,9799,c1cnc2c(c1)ccc1cccnc12,DGEZNRSVGBDHLK,<rdkit.Chem.rdchem.Mol object at 0x742d2df34ea0>,c1cnc2c(c1)ccc1cccnc12,C12H8N2,9799.0,29397.0,[M+H]+,FT,...,mona23,KO003773,NaN,3,NaN,NaN,181.0,"[(63.3, 0.051056), (73.0, 0.032188), (76.8, 0....",NaN,194629


In [16]:
result = plastchem_pred_nist_join
result.columns

Index(['plastchem_mol_id', 'smiles', 'inchikey_s', 'mol', 'scaffold',
       'formula', 'group_id', 'spec_id', 'prec_type', 'inst_type', 'frag_mode',
       'spec_type', 'ion_mode', 'dset', 'dset_spec_id', 'ace', 'prec_mz',
       'input_peaks', 'pred_mzs', 'pred_ints', 'nist20_mol_id',
       'spec_id_nist_spec', 'mol_id', 'prec_type_nist_spec',
       'inst_type_nist_spec', 'frag_mode_nist_spec', 'spec_type_nist_spec',
       'ion_mode_nist_spec', 'dset_nist_spec', 'dset_spec_id_nist_spec',
       'col_gas', 'res', 'ace_nist_spec', 'nce', 'prec_mz_nist_spec', 'peaks',
       'ri', 'group_id_nist_spec'],
      dtype='object')

In [ ]:
result = result[['plastchem_mol_id', 'nist20_mol_id', 'dset',
                 'smiles', 'inchikey_s', 'mol', 'scaffold', 'formula',
                 '']]